In [1]:
# Uncomment and run once to install dependencies
!pip install langchain langchain-community langchain-chroma langchain-groq langchain-huggingface pypdf sentence-transformers python-dotenv

  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached pyyaml-6.0.3-cp314-cp314-win_amd64.whl.metadata (2.4 kB)
  Using cached jsonpointer-3.1.1-py3-none-any.whl.metadata (2.4 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached certifi-2026.6.17-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached idna-3.18-py3-none-any.whl.metadata (6.1 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.46.4-cp314-cp314-win_amd64.whl.metadata (6.7 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata 

In [1]:
import sys
print(sys.executable)

C:\Users\saisu\Documents\AI Engineer - Sai\RAG_Multimodal_AIAgent_by_code_basics\tutorial-agentic-ai\.venv\Scripts\python.exe


In [6]:
import os
from dotenv import load_dotenv

load_dotenv()

#os.environ("GROQ_API_KEY") = "AJHVCJHADBCKJ"

assert os.getenv("GROQ_API_KEY"),"Set GROQ_API_KEY in a .env file or above"
print("API Key is loaded.")



API Key is loaded.


---
## Step 1 — Load the PDF

`PyPDFLoader` reads every page of the PDF and returns a list of `Document` objects.  
Each `Document` has a `.page_content` string (the raw text of that page).

At this stage, pages are often too long to embed efficiently — that's what Step 2 fixes.

In [7]:
from langchain_community.document_loaders import PyPDFLoader
PDFPATH = "telecom_guide.pdf"
loader = PyPDFLoader(PDFPATH)
pages = loader.load()

print(f"Loaded {len(pages)} pages from the PDF")
print(f"\n ----First page preview(first 500 character)-----")
print(pages[0].page_content[:500])

C:\Users\saisu\AppData\Local\Temp\ipykernel_26776\404832321.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 9 pages from the PDF

 ----First page preview(first 500 character)-----
Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1


In [8]:
print(pages[0].page_content[:500])

Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1


---
## Step 2 — Split into Chunks

**Why chunk?**  
- Embedding models have a token limit (e.g. 512 tokens for all-MiniLM)
- Smaller, focused chunks → better semantic similarity scores
- We want to retrieve the *relevant paragraph*, not a whole page

**Key parameters:**
- `chunk_size` — max characters per chunk
- `chunk_overlap` — characters shared between neighbouring chunks (prevents losing context at boundaries)

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 600,
    chunk_overlap = 150,
    separators = ["\n\n","\n","."," "],
)

chunks = splitter.split_documents(pages)

print(f"Total chunks : {len(chunks)} (from {len(pages)} pages)")
print(f"Avg chunk length : {sum(len(c.page_content) for c in chunks) // len(chunks)} chars")
print("\n----Example chunk ---")
print(chunks[5].page_content)

Total chunks : 44 (from 9 pages)
Avg chunk length : 495 chars

----Example chunk ---
Telecom Technical Reference Guide  - Internal Use Only
2. Troubleshooting Connectivity Issues
Connectivity problems are the most common category of customer complaints. A structured diagnostic approach
resolves the majority of cases without escalation.
Step 1  - Verify signal strength. Open the device's status bar or dial *3001#12345#* (iOS) or use a network signal
app (Android) to view the raw signal level in dBm. A signal above -85 dBm is good; between -85 and -100 dBm is
marginal; below -100 dBm is poor. If signal is weak, moving closer to a window or to a higher floor often helps.


---
## Step 3 — Embed Chunks and Store in a Vector Database

**What is an embedding?**  
A numerical vector (list of floats) that represents the *meaning* of a piece of text.  
Texts with similar meaning have vectors that are close together in high-dimensional space.

**What is a vector store?**  
A database optimised for *similarity search* — given a query vector, find the nearest stored vectors.

Here we use:
- **Embedding model:** `all-MiniLM-L6-v2` — a small, fast, open-source sentence transformer (384 dimensions)
- **Vector store:** Chroma — an in-memory (or persisted) vector DB

In [10]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

print("Loading embedding model (downloads ~90 MB on first run).....")
embeddings = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")

print("Embedding all chunks and storing in Chroma(in memory).....")
vector_store = Chroma.from_documents(chunks,embeddings)

print(f"Vector store ready .{vector_store._collection.count()} vectors stored.")

Loading embedding model (downloads ~90 MB on first run).....


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3283.90it/s]


Embedding all chunks and storing in Chroma(in memory).....
Vector store ready .44 vectors stored.


---
## Step 4 — Test the Retriever

Before hooking up the LLM, let's verify the retriever works.  
It embeds the query with the same model, then finds the top-k nearest chunk vectors.

In [11]:
retriever = vector_store.as_retriever(search_kwargs = {"k": 3})

test_query = "What is VoLTE and how does it improve call quality?"
retrieved = retriever.invoke(test_query)

print(f"Query : {test_query}")
print(f"Retrieved {len(retrieved)} chunks: \n")
for i,doc in enumerate(retrieved,1):
    print(f"------chunk{i}-----")
    print(doc.page_content[:300])
    print()

Query : What is VoLTE and how does it improve call quality?
Retrieved 3 chunks: 

------chunk1-----
legacy calls), faster call setup times (under 2 seconds versus 6-8 seconds on 3G), and the ability to use data and
voice simultaneously without degradation. VoLTE requires a compatible device, a VoLTE-enabled SIM, and an
account that has VoLTE activated.
Enabling VoLTE: On most Android devices navig

------chunk2-----
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls 

------chunk3-----
the device may not support VoLTE or the profile has not been pushed to the SIM. Agents can push the VoLTE
profile remotely via the subscriber management system.
VoWiFi (Wi-Fi Calling): VoWiFi extends IMS calling over any Wi-Fi network, including home broadband

---
## Step 5 — Build the RAG Chain

Now we wire everything together using LangChain's **LCEL (LangChain Expression Language)**:

```
question
   │
   ├──► retriever ──► join chunks into one string ──►  context
   │                                                        │
   └────────────────────────────────────────────►  prompt template
                                                            │
                                                           LLM
                                                            │
                                                       Answer (string)
```

The system prompt tells the LLM to *only* use the retrieved context — this prevents hallucination.

In [12]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

def format_docs(docs):
    return"\n\n---\n\n".join(doc.page_content for doc in docs)

SYSTEM_PROMPT = """\
You are a helpful telecom assistant.
Answer the question using ONLY the context provided below.
If the context does not contain enough information, say so clearly.

Context:
{context}
"""
prompt = ChatPromptTemplate.from_messages([
    ("system",SYSTEM_PROMPT),
    ("human","{question}"),
    ])
llm= ChatGroq(
    model = "qwen/qwen3-32b",
    temperature = 0,
    reasoning_format = "parsed",
)

chain = (
    {"context" : retriever|format_docs,"question":RunnablePassthrough()}
    |prompt
    |llm
    |StrOutputParser()
)

print("RAG Chain assembled.")

RAG Chain assembled.


---
## Step 6 — Ask a Single Question

Let's test the full pipeline end-to-end with one hard-coded question:

In [14]:
question = "How does international roaming work and what charges should I expect?"
print(f"Q : {question}\n")
print("A:",chain.invoke(question))

Q : How does international roaming work and what charges should I expect?

A: International roaming allows your device to connect to partner networks in foreign countries when you travel outside your home network's coverage. Here's how it works and the charges you may incur:

### **How It Works**  
1. **Network Connection**: When abroad, your device automatically connects to a partner network in the visited country.  
2. **Authentication**: The foreign network verifies your subscription via inter-operator protocols (SS7/Diameter), and your home network authorizes service.  
3. **Activation Requirement**: Roaming must be enabled **before departure** via the MyTelecom app (Plan & Services > International Roaming) or by calling 611. Activation takes up to 15 minutes.  

### **Charges**  
- **Zone A** (e.g., Japan, Singapore): Moderate per-MB/minute rates.  
- **Zone B** (selected countries): Similar moderate rates.  
- **Zone C** (Rest of World): Highest per-MB/minute charges.  

**Bundle

---
## Step 7 — Interactive Q&A Loop

Run this cell and keep asking questions.  
Type `quit` to exit.

> **Try these questions:**
> - "What is the difference between 4G and 5G?"
> - "How do I fix a SIM card that is not being detected?"
> - "What is SIM swap fraud and how can I protect myself?"
> - "Explain VoLTE in simple terms."
> - "What happens during a billing dispute?"

In [15]:
print("Telecom RAG Assistant - type 'quit' to exit \n")

while True:
    question = input("Your question: ").strip()
    if question.lower() in ("quit","exit","q"):
        print("Goodbye!")
        break
    if not question : 
        continue
    print("\n Answers : ")
    for chunk in chain.stream(question):
        print(chunk,end = "",flush = True)
    print("\n")
    

Telecom RAG Assistant - type 'quit' to exit 



Your question:  What is the difference between 4G and 5G?"



 Answers : 
                                                                                                                                                                                                                                                   The  key  differences  between   4 G  and   5 G  are :   

 1 .  ** Speed **:   
     -   4 G  ( LTE )  offers  typical  download  speeds  of  ** 2 0 – 1 5 0  Mbps **,  with  LTE - Advanced  pushing  speeds  beyond  ** 3 0 0  Mbps ** .   
     -   5 G  targets  ** peak  speeds  up  to   1 0  G bps **,  significantly  faster  than   4 G .   

 2 .  ** Lat ency **:   
     -   4 G  has  latency  ** below   5 0  ms ** .   
     -   5 G  reduces  latency  to  ** under   1  ms **,  critical  for  real -time  applications  like  autonomous  vehicles .   

 3 .  ** Device  Density **:   
     -   4 G  supports  standard  mobile  broadband .   
     -   5 G  can  connect  ** up  to   1  million  devices  per  square  kilomet re **,  enabling 

Your question:  How do I fix a SIM card that is not being detected?



 Answers : 
                                                                                                                                                                                                                                                                                                                                                  To  fix  a  SIM  card  not  being  detected ,  follow  these  steps  based  on  the  provided  context :

 1 .  ** Inspect  the  SIM **:   
     -  Power  off  the  device ,  remove  the  SIM ,  clean  the  gold  contacts  with  a  dry  cloth ,  and  re seat  it  firmly .  A  partially  seated  or  dirty  SIM  can  cause  detection  issues .

 2 .  ** Check  for  Network  Out age **:   
     -  Consult  the  carrier ’s  live  status  page  or  app .  If  an  outage  is  confirmed ,  wait  for  resolution  as  no  device -level  action  will  restore  service .

 3 .  ** Test  with  Another  SIM  or  Device **:   
     -  Swap  the  SIM  into

Your question:  q


Goodbye!


---
## Bonus — Inspect What Was Retrieved

This cell shows you exactly which chunks were retrieved for a given question, before the LLM sees them.  
Great for debugging retrieval quality.

In [18]:
debug_question = "What security measures protect against SIM swap fraud?"

docs = retriever.invoke(debug_question)

print(f"Question : {debug_question}")
print(f"Retrieved {len(docs)} chunks :\n")
for i,doc in enumerate(docs,1):
    print(f"{'='*60}")
    print(f"Chunk {i} (page {doc.metadata.get('page','?')})")
    print(f"{'=' * 60}")
    print(doc.page_content)
    print()

print("\nFinal Answer:")
print(chain.invoke(debug_question))

Question : What security measures protect against SIM swap fraud?
Retrieved 3 chunks :

Chunk 1 (page 8)
intercept SMS messages (undermining SMS-based 2FA), track device location, and redirect calls. Carriers
mitigate this with SS7 firewalls and anomaly detection systems, but the risk cannot be fully eliminated on legacy
protocols. 5G's use of HTTPS-based APIs (Service Based Architecture) substantially reduces this attack surface.
SIM Swap Fraud: Described in Section 5. Key mitigation: enforce strict in-person or multi-factor remote identity
verification before any SIM replacement. Flag accounts with recent SIM swaps for elevated fraud monitoring for
30 days.

Chunk 2 (page 5)
authentication, the network and SIM independently compute a response to a random challenge using Ki and the
MILENAGE algorithm. This mutual authentication prevents cloning and man-in-the-middle attacks. SIM PINs
provide an additional layer of protection; after three incorrect PIN attempts the SIM is locked and re